# Regression analysis (secondary analysis)

For each of the 12 primary outcomes, one regression model with group, age group, gender, continent and familiarity with AI.

- Yes/no outcomes: logistic regression (Q10: "Yes" versus other answers; Q15: "No" versus other answers).
- Ordered outcomes: ordinal logistic regression (proportional odds). If the proportional odds assumption clearly fails, a logistic model on a split fixed before fitting is used instead.
- The overall effect of group (likelihood ratio test, 2 df) is corrected by Holm across the 12 models. Age, gender, continent and familiarity are secondary and get Benjamini-Hochberg adjusted p-values across 48 tests.
- Models with fewer than 70 participants in the smaller outcome group are labelled "less stable".

Reference categories: patient, age 18-24, male, Asia. Familiarity is a number from 1 to 5. Results are adjusted odds ratios (OR) with 95% confidence intervals; they show associations, not causes.

**Reads:** `data/survey_clean.csv`

**Writes to `results/`:** `model_summary.csv`, `odds_ratios.csv`, `term_tests.csv`, `proportional_odds_check.csv`, `familiarity_form_check.csv`

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests

DATA_FILE = Path("../data/survey_clean.csv")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

ALPHA = 0.05

OUTCOMES = [  # (number, label, kind, function giving the outcome)
    (1,  "Q7 bias concern",                          "ordinal", lambda d: d["bias_concern_code"]),
    (2,  "Q8 privacy is protected",                  "ordinal", lambda d: d["privacy_protection_code"]),
    (3,  "Q9 explanations important",               "ordinal", lambda d: d["explanation_importance_code"]),
    (4,  "Q10 inform patients: Yes",                 "binary",  lambda d: (d["inform_patients"] == "Yes").astype(int)),
    (5,  "Q11 accountable: AI developers",           "binary",  lambda d: d["acc_ai_developers"]),
    (6,  "Q11 accountable: healthcare professional", "binary",  lambda d: d["acc_healthcare_professional"]),
    (7,  "Q11 accountable: hospital or clinic",      "binary",  lambda d: d["acc_hospital_clinic"]),
    (8,  "Q12 trust",                                "ordinal", lambda d: d["trust_code"]),
    (9,  "Q13 autonomy could be undermined",         "ordinal", lambda d: d["autonomy_undermined_code"]),
    (10, "Q14 guidelines are enough",                "ordinal", lambda d: d["guidelines_confidence_code"]),
    (11, "Q15 comfortable with no human: No",        "binary",  lambda d: (d["no_human_life_death"] == "No").astype(int)),
    (12, "Q16 disparities concern",                  "ordinal", lambda d: d["disparities_concern_code"]),
]

# Split used if the proportional odds assumption clearly fails (fixed before fitting).
PO_SPLIT = {2: 4, 9: 4, 1: 3, 3: 3, 10: 3, 12: 3}   # agree scales: code 4-5; Q7/Q9/Q14/Q16: code 3-5 (somewhat or higher)
PO_SPLIT_LABEL = {2: "agree or strongly agree", 9: "agree or strongly agree",
                  1: "any concern", 3: "any importance", 10: "any confidence", 12: "any concern"}

TERMS = {
    "group": ["group_healthcare_professional", "group_ai_developer"],
    "age": ["age_25_34", "age_35_plus"],
    "gender": ["female"],
    "continent": ["continent_not_asia"],
    "familiarity": ["familiarity"],
}
LABELS = {
    "group_healthcare_professional": "Healthcare professional (vs patient)",
    "group_ai_developer": "AI developer (vs patient)",
    "age_25_34": "Age 25-34 (vs 18-24)",
    "age_35_plus": "Age 35 and over (vs 18-24)",
    "female": "Female (vs male)",
    "continent_not_asia": "Not Asia (vs Asia)",
    "familiarity": "Familiarity (per step, 1-5)",
}

## Load the data and build the predictors

In [2]:
df = pd.read_csv(DATA_FILE)
assert set(df["gender"]) == {"Male", "Female"}

X = pd.DataFrame(index=df.index)
X["group_healthcare_professional"] = (df["group"] == "Healthcare professional").astype(float)
X["group_ai_developer"] = (df["group"] == "AI developer").astype(float)
X["age_25_34"] = (df["age_group"] == "25-34").astype(float)
X["age_35_plus"] = df["age_group"].isin(["35-44", "45-54", "55-64", "65 and above"]).astype(float)
X["female"] = (df["gender"] == "Female").astype(float)
X["continent_not_asia"] = (df["continent"] != "Asia").astype(float)
X["familiarity"] = df["familiarity_code"].astype(float)
assert X.notna().all().all() and X.shape[1] == 7

FAM_CATS = pd.get_dummies(df["familiarity_code"].astype(int), prefix="fam", drop_first=True).astype(float)
print("Participants:", len(df), "| predictors:", list(X.columns))

Participants: 253 | predictors: ['group_healthcare_professional', 'group_ai_developer', 'age_25_34', 'age_35_plus', 'female', 'continent_not_asia', 'familiarity']


## Size rule

Yes/no outcomes: the smaller of the two outcome groups. Ordered outcomes with five answers: the two answers at each end are merged and the smaller end is used. 70 or more: normal; 35 to 69: less stable; under 35: not fitted. Trust has only three answers and is fitted and labelled less stable.

In [3]:
def size_check(y, kind, num):
    if kind == "binary":
        smaller = int(min(y.sum(), len(y) - y.sum()))
    else:
        counts = y.value_counts().sort_index()
        smaller = int(min(counts.iloc[:2].sum(), counts.iloc[-2:].sum())) if len(counts) == 5 else int(counts.min())
    if smaller >= 70:
        return smaller, "normal"
    if smaller >= 35 or num == 8:   # trust
        return smaller, "less stable"
    return smaller, "not fitted"


sizes = pd.DataFrame([{"outcome_no": num, "outcome": label, "kind": kind,
                       **dict(zip(["smaller_group", "stability"], size_check(fn(df), kind, num)))}
                      for num, label, kind, fn in OUTCOMES])
sizes

,outcome_no,outcome,kind,smaller_group,stability
0,1,Q7 bias concern,ordinal,75,normal
1,2,Q8 privacy is protected,ordinal,51,less stable
2,3,Q9 explanations important,ordinal,58,less stable
3,4,Q10 inform patients: Yes,binary,64,less stable
4,5,Q11 accountable: AI developers,binary,92,normal
5,6,Q11 accountable: healthcare professional,binary,81,normal
6,7,Q11 accountable: hospital or clinic,binary,115,normal
7,8,Q12 trust,ordinal,28,less stable
8,9,Q13 autonomy could be undermined,ordinal,39,less stable
9,10,Q14 guidelines are enough,ordinal,37,less stable


## Model functions

In [4]:
def fit(y, Xm, kind):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if kind == "binary":
            res = sm.Logit(y, sm.add_constant(Xm)).fit(disp=0, maxiter=200)
        else:
            res = OrderedModel(pd.Series(pd.Categorical(y, ordered=True)), Xm, distr="logit").fit(
                method="bfgs", disp=False, maxiter=5000)
    return res, res.mle_retvals["converged"]


def lr_test(full, reduced, df_diff):
    stat = 2 * (full.llf - reduced.llf)
    return float(stat), float(stats.chi2.sf(stat, df_diff))

## Proportional odds check (before any group test)

A logistic model is fitted at each cut-point of the ordered scale (cut-points with fewer than 10 participants on one side are skipped). The assumption clearly fails if, for any predictor, the 95% confidence intervals of two cut-points do not overlap. Failing outcomes are replaced by the fixed split before the group tests.

In [5]:
def po_check(y):
    cut_cis = {}
    for k in sorted(y.unique())[1:]:
        yb = (y >= k).astype(int)
        if min(yb.sum(), len(yb) - yb.sum()) < 10:
            continue
        cut_cis[k] = fit(yb, X, "binary")[0].conf_int()
    fails = set()
    for col in X.columns:
        cis = [ci.loc[col].values for ci in cut_cis.values()]
        for i in range(len(cis)):
            for j in range(i + 1, len(cis)):
                if cis[i][1] < cis[j][0] or cis[j][1] < cis[i][0]:
                    fails.add(LABELS[col])
    return len(cut_cis), sorted(fails)


rows = []
for num, label, kind, fn in OUTCOMES:
    if kind != "ordinal":
        continue
    n_cuts, fails = po_check(fn(df).astype(int))
    rows.append({"outcome_no": num, "outcome": label, "cut_points_checked": n_cuts,
                 "predictors_failing": "; ".join(fails) or "none",
                 "assumption": "clearly fails" if fails else "no clear failure",
                 "action": "replaced by logistic model: " + PO_SPLIT_LABEL[num] + " vs rest" if fails else "ordinal model kept"})
po = pd.DataFrame(rows)

for num in po.loc[po["assumption"] == "clearly fails", "outcome_no"]:
    i = [o[0] for o in OUTCOMES].index(num)
    _, label, _, fn = OUTCOMES[i]
    OUTCOMES[i] = (num, label + ": " + PO_SPLIT_LABEL[num], "binary",
                   lambda d, fn=fn, cut=PO_SPLIT[num]: (fn(d) >= cut).astype(int))
    smaller, stab = size_check(OUTCOMES[i][3](df), "binary", num)
    sizes.loc[sizes["outcome_no"] == num, ["outcome", "kind", "smaller_group", "stability"]] = [OUTCOMES[i][1], "binary", smaller, stab]
po

,outcome_no,outcome,cut_points_checked,predictors_failing,assumption,action
0,1,Q7 bias concern,4,Not Asia (vs Asia),clearly fails,replaced by logistic model: any concern vs rest
1,2,Q8 privacy is protected,4,"Familiarity (per step, 1-5)",clearly fails,replaced by logistic model: agree or strongly ...
2,3,Q9 explanations important,3,Age 25-34 (vs 18-24),clearly fails,replaced by logistic model: any importance vs ...
3,8,Q12 trust,2,none,no clear failure,ordinal model kept
4,9,Q13 autonomy could be undermined,4,none,no clear failure,ordinal model kept
5,10,Q14 guidelines are enough,3,Age 35 and over (vs 18-24),clearly fails,replaced by logistic model: any confidence vs ...
6,12,Q16 disparities concern,4,none,no clear failure,ordinal model kept


## Fit the 12 models

In [6]:
summary_rows, or_rows, term_rows = [], [], []
fits = {}
for (num, label, kind, fn), (_, srow) in zip(OUTCOMES, sizes.iterrows()):
    if srow["stability"] == "not fitted":
        summary_rows.append({"outcome_no": num, "outcome": label, "model": "not fitted", "stability": "not fitted"})
        continue
    y = fn(df).astype(int)
    res, conv = fit(y, X, kind)
    fits[num] = (res, y, kind)
    for term, cols in TERMS.items():
        reduced, _ = fit(y, X.drop(columns=cols), kind)
        stat, p = lr_test(res, reduced, len(cols))
        term_rows.append({"outcome_no": num, "outcome": label, "term": term, "lr_chi2": stat, "df": len(cols), "p": p})
    ci = res.conf_int()
    for col in X.columns:
        or_rows.append({"outcome_no": num, "outcome": label, "predictor": LABELS[col],
                        "or": float(np.exp(res.params[col])), "ci_low": float(np.exp(ci.loc[col, 0])),
                        "ci_high": float(np.exp(ci.loc[col, 1])), "p": float(res.pvalues[col])})
    summary_rows.append({"outcome_no": num, "outcome": label,
                         "model": "logistic" if kind == "binary" else "ordinal logistic",
                         "n": int(res.nobs), "events_or_smaller_group": int(srow["smaller_group"]),
                         "stability": srow["stability"], "converged": bool(conv),
                         "log_likelihood": float(res.llf), "aic": float(res.aic)})

summary = pd.DataFrame(summary_rows)
odds = pd.DataFrame(or_rows)
terms = pd.DataFrame(term_rows)
assert summary["converged"].all() and all(int(r[0].nobs) == len(df) for r in fits.values())
print("Models fitted:", len(fits), "| all converged")

Models fitted: 12 | all converged


## Group effect (Holm across 12) and other characteristics (Benjamini-Hochberg across 48)

In [7]:
group_tests = terms[terms["term"] == "group"].copy()
group_tests["p_holm"] = multipletests(group_tests["p"], method="holm")[1]
group_tests["difference_after_holm"] = group_tests["p_holm"] < ALPHA
summary = summary.merge(group_tests[["outcome_no", "lr_chi2", "p", "p_holm", "difference_after_holm"]]
                        .rename(columns={"lr_chi2": "group_lr_chi2", "p": "group_p", "p_holm": "group_p_holm"}),
                        on="outcome_no", how="left")

other = terms[terms["term"] != "group"].copy()
other["p_bh_48"] = multipletests(other["p"], method="fdr_bh")[1]
terms = pd.concat([group_tests.drop(columns=["difference_after_holm"]), other], ignore_index=True)

print("Outcomes with an adjusted group difference after Holm:", int(summary["difference_after_holm"].sum()), "of 12")
print("Other characteristics with Benjamini-Hochberg p < 0.05:")
print(other[other["p_bh_48"] < 0.05][["outcome", "term", "lr_chi2", "p", "p_bh_48"]].round(4).to_string())
summary[["outcome_no", "outcome", "model", "stability", "group_lr_chi2", "group_p", "group_p_holm"]].round(4)

Outcomes with an adjusted group difference after Holm: 0 of 12
Other characteristics with Benjamini-Hochberg p < 0.05:
                                      outcome    term  lr_chi2       p  p_bh_48
11  Q9 explanations important: any importance     age  17.7497  0.0001   0.0067
42           Q13 autonomy could be undermined  gender  11.4797  0.0007   0.0169


,outcome_no,outcome,model,stability,group_lr_chi2,group_p,group_p_holm
0,1,Q7 bias concern: any concern,logistic,normal,0.6055,0.7388,1.0000
1,2,Q8 privacy is protected: agree or strongly agree,logistic,normal,0.7504,0.6872,1.0000
2,3,Q9 explanations important: any importance,logistic,less stable,1.1337,0.5673,1.0000
3,4,Q10 inform patients: Yes,logistic,less stable,4.0104,0.1346,1.0000
4,5,Q11 accountable: AI developers,logistic,normal,8.3048,0.0157,0.1887
5,6,Q11 accountable: healthcare professional,logistic,normal,0.5519,0.7588,1.0000
6,7,Q11 accountable: hospital or clinic,logistic,normal,1.9828,0.3711,1.0000
7,8,Q12 trust,ordinal logistic,less stable,0.8363,0.6583,1.0000
8,9,Q13 autonomy could be undermined,ordinal logistic,less stable,1.5446,0.4620,1.0000
9,10,Q14 guidelines are enough: any confidence,logistic,normal,2.4090,0.2998,1.0000


## Familiarity as a number or as categories

Familiarity as a number is compared with familiarity as categories (likelihood ratio test, 3 df). This is reported only; the main models keep familiarity as a number.

In [8]:
X_cat = pd.concat([X.drop(columns=["familiarity"]), FAM_CATS], axis=1)
rows = []
for num, (res, y, kind) in fits.items():
    res_cat, _ = fit(y, X_cat, kind)
    stat, p = lr_test(res_cat, res, FAM_CATS.shape[1] - 1)
    rows.append({"outcome_no": num, "lr_chi2": stat, "df": FAM_CATS.shape[1] - 1, "p": p,
                 "aic_number": res.aic, "aic_categories": res_cat.aic, "categories_clearly_better": p < 0.05})
fam = pd.DataFrame(rows)
fam.round(3)

,outcome_no,lr_chi2,df,p,aic_number,aic_categories,categories_clearly_better
0,1,1.665,3,0.645,359.282,363.617,False
1,2,0.228,3,0.973,348.850,354.623,False
2,3,5.761,3,0.124,259.602,259.841,False
3,4,2.037,3,0.565,279.741,283.703,False
4,5,9.511,3,0.023,336.455,332.943,True
5,6,6.372,3,0.095,324.231,323.859,False
6,7,0.972,3,0.808,344.193,349.221,False
7,8,1.578,3,0.664,354.441,358.863,False
8,9,8.752,3,0.033,622.966,620.214,True
9,10,0.947,3,0.814,355.031,360.083,False


## Save results

In [9]:
summary.to_csv(RESULTS / "model_summary.csv", index=False)
odds.to_csv(RESULTS / "odds_ratios.csv", index=False)
terms.to_csv(RESULTS / "term_tests.csv", index=False)
po.to_csv(RESULTS / "proportional_odds_check.csv", index=False)
fam.to_csv(RESULTS / "familiarity_form_check.csv", index=False)
print("Saved 5 files to", RESULTS)

Saved 5 files to ../results
